# GSPO Model Evaluation
Quick sanity check on the Llama and Qwen GSPO-trained models. Loads each model, runs test prompts, and scores dialect density on outputs.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
MODELS = {
    "Llama GSPO": {
        "model_id": "jordanpainter/dialect-llama-gspo-all",
    },
    "Qwen GSPO": {
        "model_id": "jordanpainter/dialect-qwen-gspo-all",
    },
}

# SFT baselines for comparison
SFT_MODELS = {
    "Llama SFT": "jordanpainter/DialLM-Llama-sft-all",
    "Qwen SFT":  "jordanpainter/DialLM-Qwen-sft-all",
}

SYSTEM_PROMPT = "You are a helpful assistant."
MAX_NEW_TOKENS = 256
DIALECT_MODEL = "srirag/feature-identifier"

# Set True if running on a PC with limited VRAM — loads in 4-bit (~4GB per model instead of 16GB)
LOAD_IN_4BIT = True

In [ ]:
# ── Prompts ──────────────────────────────────────────────────────────────────
TEST_PROMPTS = [
    "Write a short friendly reply in British English about making a cup of tea.",
    "Describe your morning routine in a casual, conversational way.",
    "What do you reckon is the best way to spend a Sunday afternoon?",
    "Tell me about a time you were well chuffed about something.",
    "Give me some advice on learning a new skill.",
    "What's the craic? Tell me something interesting about language.",
    "Write a short message to a mate about plans for the weekend.",
    "Explain what machine learning is, like you're chatting with a friend.",
]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_model(model_id):
    print(f"Loading {model_id} ({'4-bit' if LOAD_IN_4BIT else 'bf16'})...")
    # Load tokenizer from the model itself — it has the correct chat template
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    kwargs = dict(device_map="auto", low_cpu_mem_usage=True)
    if LOAD_IN_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.bfloat16

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    print(f"  Loaded on {model.device}")
    return model, tokenizer


def build_prompt(tokenizer, prompt, model_name):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    if "Qwen" in model_name:
        kwargs["enable_thinking"] = False
    return tokenizer.apply_chat_template(messages, **kwargs)


def generate(model, tokenizer, prompt_text, model_name):
    built = build_prompt(tokenizer, prompt_text, model_name)
    inputs = tokenizer(built, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id else []
    for tok in ["<|eot_id|>", "<|end_of_turn|>", "<end_of_turn>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid and tid != tokenizer.unk_token_id:
            eos_ids.append(tid)
    eos_ids = list(dict.fromkeys(eos_ids))

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()

In [ ]:
# ── Dialect density scorer ───────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from rewards.dialect_reward_model import DialectDensityScorer

device = "cuda" if torch.cuda.is_available() else "cpu"
scorer = DialectDensityScorer(model_path=DIALECT_MODEL, device=device)
print("Dialect scorer loaded.")

def dialect_score(texts):
    scores = scorer.score_density(texts)
    if isinstance(scores, torch.Tensor):
        scores = scores.detach().cpu().tolist()
    return [round(float(s), 4) for s in scores]

In [ ]:
# ── Run evaluation ───────────────────────────────────────────────────────────
import pandas as pd

results = []

for model_name, cfg in MODELS.items():
    model, tokenizer = load_model(cfg["model_id"])

    for prompt in TEST_PROMPTS:
        response = generate(model, tokenizer, prompt, model_name)
        density = dialect_score([response])[0]
        results.append({
            "model": model_name,
            "prompt": prompt,
            "response": response,
            "dialect_density": density,
        })
        print(f"[{model_name}] density={density:.4f} | {response[:100]}")

    # Free VRAM before loading next model
    del model
    torch.cuda.empty_cache()

df = pd.DataFrame(results)
print("\nDone.")

In [ ]:
# ── Summary stats ────────────────────────────────────────────────────────────
print(df.groupby("model")["dialect_density"].agg(["mean", "std", "min", "max"]).round(4))

In [ ]:
# ── Per-prompt comparison ────────────────────────────────────────────────────
for prompt in TEST_PROMPTS:
    subset = df[df["prompt"] == prompt]
    print(f"\nPrompt: {prompt}")
    print("-" * 80)
    for _, row in subset.iterrows():
        print(f"[{row['model']}] (density={row['dialect_density']})")
        print(row["response"])
        print()

In [ ]:
# ── Flag degenerate outputs ──────────────────────────────────────────────────
# Check for repetition or very short responses
def is_degenerate(text, min_words=10, repeat_threshold=5):
    words = text.split()
    if len(words) < min_words:
        return f"TOO SHORT ({len(words)} words)"
    # Check for repeated tokens
    for i in range(len(words) - repeat_threshold):
        if len(set(words[i:i+repeat_threshold])) == 1:
            return f"REPETITION detected: '{words[i]}' repeated"
    return None

print("Degenerate output check:")
any_bad = False
for _, row in df.iterrows():
    issue = is_degenerate(row["response"])
    if issue:
        any_bad = True
        print(f"  [{row['model']}] {issue}")
        print(f"    Prompt: {row['prompt']}")
        print(f"    Output: {row['response'][:200]}")
if not any_bad:
    print("  All outputs look fine.")

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────────
df.to_csv("gspo_evaluation_results.csv", index=False)
print("Saved to gspo_evaluation_results.csv")